
---
# RL01: Introduction to Reinforcement Learning
---

In this tutorial, we will inspect from a practical perspective the foundational algorithms of **tabular Reinforcement Learning** (RL), where the **environment** admits a **finite** number of **states** and the **agent** can choose among a **finite** set of possible **actions** in each of the states.

<center>
<img src="imgs/minecraft_interaction_loop.png" width=80% />

[*Original Image Source*](https://medium.com/student-technical-community-vit-vellore/a-brief-introduction-to-reinforcement-learning-6a74f5a61834)
</center>


The tutorial is structured as follows:
* In the **🧩 Problem Formulation** section we will learn how to represent _through code_ the elements that constitute a RL problem, i.e., the **environment** and the **policy**, and how to implement the interaction between the two.
* In the **📈 Prediction** section we will tackle the algorithms that allow to **evaluate** the **performance** of an agent in an environment and to estimate quantities of interest for the _control_ phase.
* Finally, in the **🎮 Control** section we will present the learning algorithms that allow to **optimize** the **behavior** of the agent, with the objective of learning the optimal policy for a given environment.

In this notebook you will find some **⭐ Exercise**s where you need to implement missing parts in the code. When you need to complete some code, the section is marked as:
```
# ------------------
# You code goes here
# ------------------
```

# ⚙️ Setup

Run the cell below to install and import all the notebook requirements.

In [ ]:
!pip install gymnasium
!pip install numpy
!pip install matplotlib
!git clone https://github.com/gianmtedeschi/tutorial-rlss26.git

%cd tutorial-rlss26/RL-basics

from utils import *
setup()

**Let's begin!**

# 🧩 Problem Formulation

## Environment

**Markov Decision Processes.** The fundamental model behind a RL problem is the **Markov Decision Process** (MDP), defined by:
* A finite set of **states**: $\mathcal{S}$, $|\mathcal{S}| < +\infty$.
* A finite set of **actions**: $\mathcal{A}$, $|\mathcal{A}| < +\infty$.
* A state **transition** probability **matrix**: $P(\cdot \mid s, a) \in \Delta(\mathcal{S})$.
* A **reward function**: $R(s, a) = \mathbb{E}[R_t \mid S_t = s, A_t = a]$.
* An **inistial state distribution**: $\mu_0 \in \Delta(\mathcal{S})$.

_Remark._ The transition matrix and the initial state distribution are unknown: this is where learning comes into play!

**Gymnasium.** The `gymnasium` Python library models a MDP through the `Env` (_environment_) interface.

```python
class Env:
    def step(action: ActType) -> tuple[ObsType, float]:
        ...
    
    def reset() -> ObsType:
        ...
```

The semantic of the interface is the following:
* The environment keeps track of the **current state** with an **internal attribute** `state` of type `ObsType`.
* The `step` method allows to implement the **transition matrix** $P(\cdot \mid s, a)$ and the **reward function** $R(s, a)$. In particular, it takes in input an action `action` of type `ActType` and returns the **next state** `next_state` of type `ObsType` and the **reward** `r` of type `float` according to the internal state `state` and the provided action `action`.
* The `reset` method allows to implement the initial state distribution $\mu_0$. In particular it resets the internal state of the environment and the returns the new initial state `initial_state` of type `ObsType`.

<span style="color: red; font-weight: bold;">TODO:</span> add what is missing to the `step` return.

_Remark._ The `Env` interface allows to model the more general _Partially Observable_ MDP (POMDP), where the information provided to the agent is an element of the set of _observations_ $\mathcal{O}$ which, in general, can differ from the set of states $\mathcal{S}$. The class `ObsType` represents an element of the set $\mathcal{O}$. In this notebook, we will consider just the standard MDP case, where $\mathcal{O} = \mathcal{S}$. Thus `ObsType` is the class which represents the states of the MDP.

**📝 Example.** The _cliff walking_ environment models a grid world in which the agent has to learn to reach a goal position from a starting state, avoiding falling off the cliff.

<center>
<div style="position: relative; width: 900px; height: 300px; overflow: hidden; border: 1px solid #ccc;">
  <img src="https://gymnasium.farama.org/_images/cliff_walking.gif" style="position: absolute; top: 0; left: 0; width: 100%; height: 100%; object-fit: cover;" />
  <img src="imgs/cliff_walking_coords.svg" style="position: absolute; top: 0; left: 0; width: 100%; height: 100%; opacity: 0.9;" />
</div>
</center>

The environment models the following MDP:
* The **state** space is the set of pairs $\mathcal{S} = \{ 0, \dots, 3 \} \times \{ 0, \dots, 11 \}$, representing the current position of the agent in the $4 \times 12$ grid.
* The agent can do one of the following **actions**: $\mathcal{A} = \{$ Up, Right, Down, Left $\}$ represented with the numbers from $0$ to $3$ in the given order.
* The **state transition** is determinisitc: the agent moves in the direction given by the action, unless if there is a wall or if it falls off the cliff. The states off the cliff are _absorbing_: independently of the action, the state won't change. An episode is truncated after $500$ steps of interaction.
* The **goal** is to reach the state $(3, 11)$, i.e., the bottom right cell, without falling off the cliff, being as fast as possible. For this reason, the **reward** function is shaped as follows. The agent receives $-1$ reward until it reaches the goal. If it falls off the cliff, it receives $-100$ reward.
* The agent **starts** from a **random state** above the cliff.

Below, you can find an **implementation** of the environment.

```python
class CliffWalkingEnv(Env):
    ROWS = 4
    COLS = 12
    CLIFF = {(3, c) for c in range(1, 11)}
    GOAL = (3, 11)

    def reset(self, *, seed=None, options=None):
        valid_states = [
            (r, c)
            for r in range(self.ROWS)
            for c in range(self.COLS)
            if (r, c) not in self.CLIFF and (r, c) != self.GOAL
        ]
        # Sample uniformly the initial state
        state = valid_states[np.random.randint(len(valid_states))]
        self.state = np.array(state, dtype=np.int32)
        # Count the number of interactions
        self.interactions = 0
        return self.state, {}

    def step(self, action: int):
        # If possible perform the action
        if action == 0 and self.state[0] > 0:
            self.state = self.state + np.array([-1, 0], dtype=np.int32)
        elif action == 1 and self.state[1] < self.COLS - 1:
            self.state = self.state + np.array([0, 1], dtype=np.int32)
        elif action == 2 and self.state[0] < self.ROWS - 1:
            self.state = self.state + np.array([1, 0], dtype=np.int32)
        elif action == 3 and self.state[1] > 0:
            self.state = self.state + np.array([0, -1], dtype=np.int32)

        self.interactions += 1

        terminated = True if tuple(self.state) == self.GOAL else False
        truncated = self.interactions >= 500

        reward = -100 if tuple(self.state) in self.CLIFF else -1

        # When the agent falls off the cliff, terminate the episode
        if reward == -100:
           terminated = True

        return self.state, reward, terminated, truncated, {}

```

## Policy

# Interaction Loop

# 📈 Prediction

# 🎮 Control